# Baselines — Telco Customer Churn

Treinamento e avaliação dos modelos baseline com rastreamento via MLflow.

**Modelos:** DummyClassifier, Logistic Regression, Decision Tree, Random Forest  
**Validação:** StratifiedKFold (5 folds)  
**Métricas:** AUC-ROC, PR-AUC, F1, Precision, Recall

## 0. Configuração inicial — Seed e bibliotecas

In [1]:
import random
import hashlib
import logging

import numpy as np
import pandas as pd
import mlflow

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, average_precision_score

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logger.info("Seed fixado: %d", SEED)

INFO - Seed fixado: 42


## 1. Carregando dados tratados

In [2]:
DATA_PATH = "../data/telco_churn_clean.csv"

df = pd.read_csv(DATA_PATH)

# Versão do dataset via hash MD5 — para rastreabilidade no MLflow
with open(DATA_PATH, "rb") as f:
    dataset_hash = hashlib.md5(f.read()).hexdigest()

logger.info("Dataset carregado: %d linhas, %d colunas", df.shape[0], df.shape[1])
logger.info("Dataset MD5: %s", dataset_hash)

INFO - Dataset carregado: 7043 linhas, 33 colunas


INFO - Dataset MD5: 8bdf2e12867b9a60298ad9c28c5ea9bf


## 2. Preparação das features

Removendo colunas identificadoras e com vazamento de dados.
Variável alvo: `Churn Label` → 1 (Yes) / 0 (No).

In [3]:
drop_cols = ['CustomerID', 'Country', 'State', 'City', 'Lat Long',
             'Churn Reason', 'Churn Score', 'Churn Value', 'CLTV']

X = df.drop(columns=drop_cols + ['Churn Label'])
# ALVO - transformando 1 ou 0
y = (df['Churn Label'] == 'Yes').astype(int)

# FEATURES - transformando em dummies ( text em numeros )
X = pd.get_dummies(X)

logger.info("Features: %d colunas | Positivos (churn): %d (%.1f%%)",
            X.shape[1], y.sum(), y.mean() * 100)

INFO - Features: 50 colunas | Positivos (churn): 1869 (26.5%)


## 3. Configuração do MLflow

In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("telco-churn-baselines")
logger.info("MLflow configurado — experimento: telco-churn-baselines")

2026/05/04 22:17:15 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-baselines' does not exist. Creating a new experiment.


INFO - MLflow configurado — experimento: telco-churn-baselines


## 4. Treino e avaliação dos baselines

Validação cruzada estratificada com 5 folds e 5 métricas para todos os modelos.

**Métricas técnicas:**
- **AUC-ROC** — capacidade discriminativa geral
- **PR-AUC** (Average Precision) — especialmente relevante para datasets desbalanceados, pois foca na classe positiva (churn)
- **F1** — média harmônica de Precision e Recall
- **Precision** — dos que o modelo classificou como churn, quantos realmente eram
- **Recall** — dos que realmente cancelaram, quantos o modelo capturou

**Modelos:**
- **Dummy** — baseline aleatório (estratégia: classe majoritária)
- **Logistic Regression** — baseline linear com normalização
- **Decision Tree** — baseline não-linear, profundidade limitada para evitar overfitting
- **Random Forest** — ensemble de árvores, combinando múltiplas árvores para melhor generalização

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = {
    'roc_auc':           'roc_auc',
    'average_precision': 'average_precision',
    'f1':                make_scorer(f1_score),
    'precision':         make_scorer(precision_score, zero_division=0),
    'recall':            make_scorer(recall_score),
}

modelos = [
    ('Dummy',               DummyClassifier(strategy='most_frequent', random_state=SEED)),
    ('Logistic Regression', Pipeline([
                                ('scaler', StandardScaler()),
                                ('model', LogisticRegression(max_iter=1000, random_state=SEED))
                            ])),
    ('Decision Tree',       DecisionTreeClassifier(max_depth=5, random_state=SEED)),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   random_state=SEED, n_jobs=-1)),
]

print("=" * 75)
print(f"{'Modelo':<25} {'AUC':>7} {'PR-AUC':>7} {'F1':>7} {'Precision':>10} {'Recall':>7}")
print("=" * 75)

for nome, modelo in modelos:
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring)

    auc       = scores['test_roc_auc'].mean()
    pr_auc    = scores['test_average_precision'].mean()
    f1        = scores['test_f1'].mean()
    precision = scores['test_precision'].mean()
    recall    = scores['test_recall'].mean()

    with mlflow.start_run(run_name=nome):
        mlflow.log_param("model", nome)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("dataset_version", dataset_hash)
        mlflow.log_metric("auc_roc",   auc)
        mlflow.log_metric("pr_auc",    pr_auc)
        mlflow.log_metric("f1",        f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall",    recall)

    logger.info("%s — AUC: %.3f | PR-AUC: %.3f | F1: %.3f | Precision: %.3f | Recall: %.3f",
                nome, auc, pr_auc, f1, precision, recall)
    print(f"{nome:<25} {auc:>7.3f} {pr_auc:>7.3f} {f1:>7.3f} {precision:>10.3f} {recall:>7.3f}")

print("=" * 75)

Modelo                        AUC  PR-AUC      F1  Precision  Recall


INFO - Dummy — AUC: 0.500 | PR-AUC: 0.265 | F1: 0.000 | Precision: 0.000 | Recall: 0.000


Dummy                       0.500   0.265   0.000      0.000   0.000


INFO - Logistic Regression — AUC: 0.858 | PR-AUC: 0.674 | F1: 0.619 | Precision: 0.665 | Recall: 0.579


Logistic Regression         0.858   0.674   0.619      0.665   0.579


INFO - Decision Tree — AUC: 0.845 | PR-AUC: 0.621 | F1: 0.582 | Precision: 0.614 | Recall: 0.562


Decision Tree               0.845   0.621   0.582      0.614   0.562


INFO - Random Forest — AUC: 0.859 | PR-AUC: 0.678 | F1: 0.598 | Precision: 0.670 | Recall: 0.540


Random Forest               0.859   0.678   0.598      0.670   0.540
